# 08 — Milestone 3 Plots

Generates the **six per-category donor co-movement panels** referenced in [`presentation/milestone3/milestone3.tex`](../presentation/milestone3/milestone3.tex).

For each of the six donor categories, a side-by-side figure shows Brent (black) against that category's donors over the **Hormuz** (left) and **Russia** (right) windows, rebased to 100 at each pre-window start. This answers Milestone 2 review comment 1 (SUTVA): the audience sees, category by category, how each donor group co-moves with Brent and how it behaves through `T0`.

Outputs → `plots/milestone3/donors_<key>_side_by_side.png`:

| Category | File |
|---|---|
| Metals | `donors_metals_side_by_side.png` |
| Agriculturals | `donors_ags_side_by_side.png` |
| Equities | `donors_equities_side_by_side.png` |
| FX | `donors_fx_side_by_side.png` |
| Rates/credit | `donors_rates_side_by_side.png` |
| Volatility | `donors_vol_side_by_side.png` |

Style mirrors the side-by-side panel in [`07_Milestone2_Plots.ipynb`](07_Milestone2_Plots.ipynb) (same `build_event_panel`, same windows). No model fits required — raw Brent + donor levels only.

**Donor categorization** uses the **19-donor shared pool** actually fed to the SCM ensemble (Metals 3, Agriculturals 3, Equities 2, FX 8, Rates/credit 2, Volatility 1), confirmed against `data/validation/donor_importance_consensus.csv`. This deliberately does **not** include Cotton or EM_Eq — the milestone2 deck's "21" figure was an overcount.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = ROOT / 'data'
PLOTS_DIR = ROOT / 'plots' / 'milestone3'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- data ----------
brent = (pd.read_csv(DATA / 'brent_spot.csv', parse_dates=['Date'])
           .rename(columns={'Date': 'date', 'Price': 'Brent'})
           .set_index('date').sort_index().dropna())
donors = pd.read_parquet(DATA / 'donors.parquet')

# ---------- donor categories: included (solid) vs excluded (dashed) ----------
# Included = the 18-donor shared/preferred pool fed to the SCM ensemble
# (confirmed against data/validation/donor_importance_consensus.csv).
# Excluded = candidates in the same category dropped by the SUTVA audit
# (oil-supply / disruption path) — shown dashed for visual contrast.
DONOR_CATEGORIES = {
    'Metals':       {'key': 'metals',   'included': ['Silver', 'Platinum', 'Gold'],
                                         'excluded': ['Copper', 'IronOre', 'Palladium']},
    'Agriculturals':{'key': 'ags',      'included': ['Coffee', 'Sugar', 'LiveCattle'],
                                         'excluded': ['Wheat', 'Corn', 'Soybeans', 'Cotton']},
    'Equities':     {'key': 'equities', 'included': ['SP500', 'Nikkei'],
                                         'excluded': ['EM_Eq']},
    'FX':           {'key': 'fx',       'included': ['AUD', 'JPY', 'CHF', 'CNY', 'INR', 'KRW', 'ZAR', 'MXN'],
                                         'excluded': ['EUR', 'GBP', 'DXY']},
    'Rates/credit': {'key': 'rates',    'included': ['TLT', 'HYG'],
                                         'excluded': ['US10Y']},
    'Volatility':   {'key': 'vol',      'included': [],
                                         'excluded': ['VIX']},
}
assert sum(len(c['included']) for c in DONOR_CATEGORIES.values()) == 18

# ---------- focal events (Hormuz left, Russia right) ----------
FOCAL_EVENTS = [
    {'name': 'Strait of Hormuz crisis', 'slug': 'hormuz',
     'T0': pd.Timestamp('2026-02-28'), 'pre_start': pd.Timestamp('2024-06-01'),
     'post_end': pd.Timestamp('2026-05-31')},
    {'name': 'Russia invades Ukraine', 'slug': 'russia',
     'T0': pd.Timestamp('2022-02-24'), 'pre_start': pd.Timestamp('2020-07-01'),
     'post_end': pd.Timestamp('2022-09-30')},
]

# 20-color palette so even FX (8 incl + 3 excl = 11 lines) stays distinguishable.
PALETTE = list(plt.get_cmap('tab20').colors)

print(f'Brent:  {brent.shape}, {brent.index.min().date()} -> {brent.index.max().date()}')
print(f'Donors: {donors.shape}')
print(f'Output: {PLOTS_DIR}')

Brent:  (9902, 1), 1987-05-20 -> 2026-06-01
Donors: (5511, 32)
Output: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3


In [2]:
def build_event_panel(event, cols):
    """Brent + selected donor cols, pre_start..post_end, rebased to 100 at first in-window date."""
    data = (brent[['Brent']].join(donors[cols], how='outer')
                            .sort_index()
                            .loc[event['pre_start']:event['post_end']])
    data = data.ffill(limit=5).dropna()
    if data.empty:
        return None
    norm = 100 * data.divide(data.iloc[0])
    norm['_days_from_T0'] = (norm.index - event['T0']).days
    return norm


def plot_category(label, cfg):
    inc, exc = cfg['included'], cfg['excluded']
    cols = inc + exc
    color_map = dict(zip(cols, PALETTE[:len(cols)]))
    # Larger figure — the graph is the slide highlight.
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.6), sharey=False)

    for ax, event in zip(axes, FOCAL_EVENTS):
        panel = build_event_panel(event, cols)
        if panel is None:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
            continue
        x = panel['_days_from_T0'].values
        # Included: solid, full opacity.
        for c in inc:
            ax.plot(x, panel[c].values, color=color_map[c], linewidth=1.8, alpha=0.9, label=c)
        # Excluded: dashed, dimmer, labelled "(excl.)".
        for c in exc:
            ax.plot(x, panel[c].values, color=color_map[c], linewidth=1.6, alpha=0.75,
                    linestyle='--', dashes=(4, 2), label=f'{c} (excl.)')
        ax.plot(x, panel['Brent'].values, color='black', linewidth=2.8, label='Brent', zorder=10)
        ax.axvline(x=0, color='black', linestyle=':', linewidth=1.4, alpha=0.7, zorder=8)
        ax.set_xlabel(f"Days from T0 ({event['T0'].date()})", fontsize=11)
        ax.set_title(f"{event['name']}\n{event['pre_start'].date()} -> {event['post_end'].date()}",
                     fontsize=12, pad=6)
        ax.grid(alpha=0.22, linestyle='-', linewidth=0.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ncol = 2 if len(cols) > 5 else 1
        ax.legend(loc='upper left', frameon=True, fontsize=9, framealpha=0.9, ncol=ncol)

    axes[0].set_ylabel('Index (pre-window start = 100)', fontsize=11)
    fig.suptitle(f'{label} — included (solid) vs excluded (dashed) donors vs Brent',
                 fontsize=14, y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out = PLOTS_DIR / f"donors_{cfg['key']}_side_by_side.png"
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out}  ({len(inc)} incl, {len(exc)} excl)")


for label, cfg in DONOR_CATEGORIES.items():
    plot_category(label, cfg)

print('\nAll six per-category panels written to', PLOTS_DIR)

Saved: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3\donors_metals_side_by_side.png  (3 incl, 3 excl)


Saved: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3\donors_ags_side_by_side.png  (3 incl, 4 excl)


Saved: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3\donors_equities_side_by_side.png  (2 incl, 1 excl)


Saved: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3\donors_fx_side_by_side.png  (8 incl, 3 excl)


Saved: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3\donors_rates_side_by_side.png  (2 incl, 1 excl)


Saved: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3\donors_vol_side_by_side.png  (0 incl, 1 excl)

All six per-category panels written to C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3


In [3]:
# ---- Headline ATT across post-event windows (for the summary slide) ----
# Source: data/validation/final_postwindow_sensitivity.csv (ensemble median + IQR per horizon).
pw = pd.read_csv(DATA / 'validation' / 'final_postwindow_sensitivity.csv')

fig, ax = plt.subplots(figsize=(11, 5.5))
styles = {
    'hormuz': dict(color='#8B0000', marker='o', label='Hormuz 2026'),
    'russia': dict(color='#1f77b4', marker='s', label='Russia 2022'),
}
for ev, st in styles.items():
    d = pw[pw.event == ev].sort_values('n_post')
    ax.fill_between(d.n_post, d.iqr_lo, d.iqr_hi, color=st['color'], alpha=0.13, zorder=1)
    ax.plot(d.n_post, d.ens_median_gap_pct, lw=2.6, marker=st['marker'], ms=7,
            color=st['color'], label=st['label'], zorder=5)
    # Merge labels for points closer than ~6 trading days (e.g. Hormuz 3m vs full).
    groups = []
    for _, r in d.iterrows():
        if groups and (r.n_post - groups[-1]['x']) < 6:
            groups[-1]['labels'].append(r.horizon)
            groups[-1]['x'], groups[-1]['y'] = r.n_post, r.ens_median_gap_pct
        else:
            groups.append({'x': r.n_post, 'y': r.ens_median_gap_pct, 'labels': [r.horizon]})
    for g in groups:
        ax.annotate(f"{'/'.join(g['labels'])}\n{g['y']:.0f}%", (g['x'], g['y']),
                    textcoords='offset points', xytext=(0, 11), fontsize=9, ha='center',
                    color=st['color'], fontweight='semibold')

ax.axhline(0, color='black', lw=0.7, alpha=0.5)
ax.set_xlabel('Post-event window length (trading days from $T_0$)', fontsize=11)
ax.set_ylabel('Ensemble-median ATT (%)', fontsize=11)
ax.set_title('Headline ATT across post-event windows  (ensemble median, shaded = IQR)', fontsize=13)
ax.set_ylim(0, 70)
ax.grid(alpha=0.25, linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10.5, loc='upper right', frameon=True, framealpha=0.9)
fig.tight_layout()

out = PLOTS_DIR / 'postwindow_sensitivity.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
print('Saved:', out)

Saved: C:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\plots\milestone3\postwindow_sensitivity.png
